In [1]:
!pip -q install -U timm
import os, gc, math, time, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, timm
from PIL import Image
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

DEV, SZ, TAG = "cuda", 336, "D3L336"
ARCH = "vit_large_patch16_dinov3.lvd1689m"
COMP = "/kaggle/input/competitions/ai-of-god-5/participant_data"
OUT  = "/kaggle/working"
EPOCHS, BS, N_UNFROZEN = 20, 8, 6
LR_TOP, LLD, LR_HEAD, WD, EMA_D = 3e-5, 0.85, 1e-3, 0.05, 0.995

df = pd.read_csv(f"{COMP}/train.csv"); te = pd.read_csv(f"{COMP}/sample_submission.csv")
le = LabelEncoder().fit(df["animal_class"]); y = le.transform(df["animal_class"]); NC = len(le.classes_)
folds = list(StratifiedKFold(5, shuffle=True, random_state=0).split(np.arange(len(y)), y))

# ---- 1) look at the raw images first ----
print(df.image_id[:3].tolist())
for iid in df.image_id[:3]:
    im = Image.open(f"{COMP}/train_images/{iid}"); a = np.asarray(im)
    print(iid, "| mode", im.mode, "| size", im.size, "| dtype", a.dtype, "| raw min/max", a.min(), a.max(),
          "| after RGB:", np.asarray(im.convert("RGB")).min(), np.asarray(im.convert("RGB")).max())

# ---- 2) robust loader (handles 16-bit / float / alpha / palette) ----
def load_rgb(path):
    im = Image.open(path)
    if im.mode in ("I;16", "I", "F"):
        a = np.asarray(im).astype(np.float32); a = a / max(a.max(), 1) * 255
        return Image.fromarray(a.astype(np.uint8)).convert("RGB")
    if im.mode in ("RGBA", "LA", "PA") or "transparency" in im.info:
        bg = Image.new("RGBA", im.size, (0, 0, 0, 255)); bg.alpha_composite(im.convert("RGBA")); return bg.convert("RGB")
    return im.convert("RGB")

def build_cache(ids, folder, path):
    tmp = path + ".tmp.npy"
    a = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.uint8, shape=(len(ids), SZ, SZ, 3))
    for i, iid in enumerate(ids):
        im = load_rgb(f"{COMP}/{folder}/{iid}"); s = SZ / max(im.size)
        im = im.resize((max(1, round(im.width*s)), max(1, round(im.height*s))), Image.BILINEAR)
        c = Image.new("RGB", (SZ, SZ)); c.paste(im, ((SZ-im.width)//2, (SZ-im.height)//2)); a[i] = np.asarray(c)
    a.flush(); del a; os.replace(tmp, path)

# ---- 3) delete stale caches and rebuild ----
ptr, pte = f"{OUT}/train_{SZ}.npy", f"{OUT}/test_{SZ}.npy"
for p_ in (ptr, pte, ptr + ".tmp.npy", pte + ".tmp.npy"):
    if os.path.exists(p_): os.remove(p_)
build_cache(df.image_id, "train_images", ptr); build_cache(te.image_id, "test_images", pte)
Xtr = np.load(ptr, mmap_mode="r"); Xte = np.load(pte, mmap_mode="r")
for nm, X in (("train", Xtr), ("test", Xte)):
    sample = np.asarray(X[np.linspace(0, len(X)-1, 40).astype(int)])
    print(nm, X.shape, "sample max", sample.max(), "| nonblack:", int((sample.reshape(40, -1).max(1) > 0).sum()), "/ 40")
    assert (sample.reshape(40, -1).max(1) > 0).all(), f"{nm} cache has all-black rows: send me the raw-image printout above"
print(NC)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 9.1 MB/s eta 0:00:00
['000dc07b2b676f5d0ec0acf9.png', '0063d33c0996495e9337062ad3f13319.png', '007227c159bdba58d63406ca.png']
000dc07b2b676f5d0ec0acf9.png | mode RGB | size (329, 153) | dtype uint8 | raw min/max 10 250 | after RGB: 10 250
0063d33c0996495e9337062ad3f13319.png | mode RGB | size (310, 163) | dtype uint8 | raw min/max 37 246 | after RGB: 37 246
007227c159bdba58d63406ca.png | mode RGB | size (1200, 627) | dtype uint8 | raw min/max 36 252 | after RGB: 36 252
train (3087, 336, 336, 3) sample max 255 | nonblack: 40 / 40
test (1179, 336, 336, 3) sample max 252 | nonblack: 40 / 40
72


In [2]:
MEAN = torch.tensor([0.485,0.456,0.406], device=DEV).view(1,3,1,1)
STD  = torch.tensor([0.229,0.224,0.225], device=DEV).view(1,3,1,1)
PERMS = [[0,1,2],[1,2,0],[2,0,1],[2,1,0]]

# ---------- GPU augmentation (color destroyed, shape kept) ----------
def curves(x, K=9):
    B, C, H, W = x.shape
    k = torch.cumsum(torch.rand(B,3,K-1, device=DEV)**2 + 1e-3, -1); k = k / k[..., -1:]
    k = torch.cat([torch.zeros(B,3,1, device=DEV), k], -1).reshape(B*C, K)
    t = x.clamp(0,1).reshape(B*C, -1) * (K-1); lo = t.floor().clamp(max=K-2).long(); fr = t - lo
    return (torch.gather(k,1,lo)*(1-fr) + torch.gather(k,1,lo+1)*fr).view(B,C,H,W)

def color_aug(x):
    B = x.shape[0]; r = torch.rand(B,1,1,1, device=DEV)
    M = torch.rand(B,3,3, device=DEV)**2 + 0.02; M = M / M.sum(-1, keepdim=True)
    xm = torch.einsum('bij,bjhw->bihw', M, x)
    lo, hi = xm.amin((2,3), keepdim=True), xm.amax((2,3), keepdim=True)
    xm = curves((xm-lo)/(hi-lo+1e-3))
    xs = torch.gather(x, 1, torch.argsort(torch.rand(B,3, device=DEV),1)[:,:,None,None].expand_as(x))
    xg = x.mean(1, keepdim=True).expand_as(x)
    out = torch.where(r<0.6, xm, torch.where(r<0.8, xs, xg))
    return torch.where(torch.rand(B,1,1,1, device=DEV)<0.15, 1-out, out)

def degrade(x):
    B = x.shape[0]; w = torch.ones(3,1,5,5, device=DEV)/25
    xb = F.conv2d(F.pad(x,(2,2,2,2),mode='reflect'), w, groups=3)
    x = torch.where(torch.rand(B,1,1,1, device=DEV)<0.3, xb, x)
    lv = 2**torch.randint(3,6,(B,1,1,1), device=DEV).float() - 1
    x = torch.where(torch.rand(B,1,1,1, device=DEV)<0.2, torch.round(x*lv)/lv, x)
    x = x + (torch.rand(B,1,1,1, device=DEV)<0.5) * torch.rand(B,1,1,1, device=DEV)*0.05 * torch.randn_like(x)
    return x.clamp(0,1)

def geom(x):
    B = x.shape[0]; u = lambda: torch.rand(B, device=DEV)*2-1
    ang = u()*math.radians(20); sc = 0.75 + torch.rand(B, device=DEV)*0.5; asp = torch.exp(u()*0.1)
    flip = (torch.rand(B, device=DEV)<0.5).float()*2-1; c, s = torch.cos(ang), torch.sin(ang)
    sx, sy = 1/(sc*asp), asp/sc; th = torch.zeros(B,2,3, device=DEV)
    th[:,0,0]=c*sx*flip; th[:,0,1]=-s*sy; th[:,0,2]=u()*0.1
    th[:,1,0]=s*sx*flip; th[:,1,1]= c*sy; th[:,1,2]=u()*0.1
    return F.grid_sample(x, F.affine_grid(th, x.shape, align_corners=False), padding_mode='zeros', align_corners=False)

def cutout(x, p=0.3, mf=0.2):
    B,_,H,W = x.shape
    for i in torch.nonzero(torch.rand(B)<p).flatten().tolist():
        h, w = int(H*mf*np.random.uniform(.3,1)), int(W*mf*np.random.uniform(.3,1))
        y0, x0 = np.random.randint(0,H-h), np.random.randint(0,W-w)
        x[i,:,y0:y0+h,x0:x0+w] = torch.rand(3,1,1, device=DEV)
    return x

def train_aug(xb):
    x = xb.to(DEV).permute(0,3,1,2).float()/255
    rows = (x.amax((1,3)) > 0); cols = (x.amax((1,2)) > 0)            # non-padded bounding box
    box = (rows[:,:,None] & cols[:,None,:])[:,None].float()
    x = degrade(color_aug(x)) * box                                     # keep letterbox black like test
    return (cutout(geom(x)) - MEAN) / STD

# ---------- model ----------
def build():
    m = timm.create_model(ARCH, pretrained=True, num_classes=NC, img_size=SZ, drop_rate=0.3)
    for p in m.parameters(): p.requires_grad = False
    groups, nb = [], len(m.blocks)
    def add(params, lr):
        params = list(params)
        for p in params: p.requires_grad = True
        groups.append(dict(params=[p for p in params if p.ndim>=2], lr=lr, weight_decay=WD))
        groups.append(dict(params=[p for p in params if p.ndim<2],  lr=lr, weight_decay=0.0))
    for i in range(nb-N_UNFROZEN, nb): add(m.blocks[i].parameters(), LR_TOP * LLD**(nb-1-i))
    for nm in ("norm", "fc_norm"):
        mod = getattr(m, nm, None)
        if mod is not None and len(list(mod.parameters())): add(mod.parameters(), LR_TOP)
    add(m.head.parameters(), LR_HEAD)
    return m, [g for g in groups if g['params']]

class EMA:
    def __init__(s, m): s.n = 0; s.sh = {k: p.detach().clone() for k,p in m.named_parameters() if p.requires_grad}
    @torch.no_grad()
    def update(s, m):
        s.n += 1; d = min(EMA_D, (1+s.n)/(10+s.n))
        for k,p in m.named_parameters():
            if k in s.sh: s.sh[k].mul_(d).add_(p.detach(), alpha=1-d)
    @torch.no_grad()
    def apply(s, m):
        for k,p in m.named_parameters():
            if k in s.sh: p.copy_(s.sh[k])

@torch.no_grad()
def predict(m, X, idx, bs=32):                       # 4 channel perms x flip = 8 passes
    m.eval(); out = []
    for i in range(0, len(idx), bs):
        x = torch.from_numpy(np.asarray(X[idx[i:i+bs]])).to(DEV).permute(0,3,1,2).float()/255; acc = 0
        for p in PERMS:
            for f in (0,1):
                v = x[:, p]; v = v.flip(3) if f else v
                with torch.autocast('cuda', dtype=torch.float16): acc = acc + m((v-MEAN)/STD).float().softmax(1)
        out.append((acc/8).cpu())
    return torch.cat(out).numpy()

# ---------- one fold ----------
def run_fold(k):
    fo, fi, ft = (f"{OUT}/{n}_{TAG}_f{k}.npy" for n in ("oof","va_idx","test"))
    if all(os.path.exists(f) for f in (fo,fi,ft)): print(f"fold {k} done already"); return
    tr, va = folds[k]; torch.manual_seed(k); np.random.seed(k)
    m, groups = build(); m.to(DEV); trainable = [p for g in groups for p in g['params']]
    opt = torch.optim.AdamW(groups); spe = len(tr)//BS
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[g['lr'] for g in groups], total_steps=EPOCHS*spe,
                                                pct_start=0.1, div_factor=25, final_div_factor=100)
    scaler, ema = torch.amp.GradScaler(), EMA(m)
    cnt = np.bincount(y[tr], minlength=NC)
    crit = nn.CrossEntropyLoss(weight=torch.tensor(len(tr)/(NC*cnt), dtype=torch.float32, device=DEV), label_smoothing=0.1)
    t0 = time.time()
    for ep in range(EPOCHS):
        m.train(); perm = np.random.permutation(tr); run = 0
        for b in range(spe):
            ii = np.sort(perm[b*BS:(b+1)*BS])
            x = train_aug(torch.from_numpy(np.asarray(Xtr[ii]))); yt = torch.from_numpy(y[ii]).to(DEV)
            with torch.autocast('cuda', dtype=torch.float16): loss = crit(m(x).float(), yt)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0); scaler.step(opt); scaler.update(); sched.step(); ema.update(m)
            run += loss.item()
        print(f"fold {k} ep {ep+1}/{EPOCHS} loss {run/spe:.4f}  {(time.time()-t0)/60:.1f} min")
    ema.apply(m)                                                    # final weights = EMA
    p_va = predict(m, Xtr, va); p_te = predict(m, Xte, np.arange(len(te)))
    np.save(fo, p_va.astype(np.float32)); np.save(fi, va); np.save(ft, p_te.astype(np.float32))
    print(f"fold {k}: acc {np.mean(p_va.argmax(1)==y[va]):.4f}  macroF1 {f1_score(y[va], p_va.argmax(1), average='macro'):.4f}")
    del m, opt, ema, scaler, sched; gc.collect(); torch.cuda.empty_cache()

# ---------- run all folds (time-guarded) ----------
T0, BUDGET_H, FOLD_H = time.time(), 10, 1.5        # set FOLD_H to your measured hours per fold
for k in range(5):
    if (time.time()-T0)/3600 + FOLD_H > BUDGET_H: print(f"stopping before fold {k}: not enough time"); break
    run_fold(k)

# ---------- OOF score + submission from finished folds ----------
oof = np.full((len(y), NC), np.nan, np.float32); tests = []
for k in range(5):
    if os.path.exists(f"{OUT}/oof_{TAG}_f{k}.npy"):
        oof[np.load(f"{OUT}/va_idx_{TAG}_f{k}.npy")] = np.load(f"{OUT}/oof_{TAG}_f{k}.npy"); tests.append(np.load(f"{OUT}/test_{TAG}_f{k}.npy"))
ok = ~np.isnan(oof[:,0])
print(f"folds done: {len(tests)} | rows {ok.sum()} | OOF macro F1 {f1_score(y[ok], oof[ok].argmax(1), average='macro'):.4f}")
if tests:
    pt = np.mean(tests, 0); np.save(f"{OUT}/test_{TAG}_mean.npy", pt)
    pd.DataFrame({"image_id": te.image_id, "animal_class": le.classes_[pt.argmax(1)]}).to_csv(f"{OUT}/submission.csv", index=False)
    print("saved submission.csv")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.21GB            

model.safetensors: downloading bytes:           |  0.00B            

fold 0 ep 1/20 loss 3.5617  1.8 min
fold 0 ep 2/20 loss 1.1920  3.7 min
fold 0 ep 3/20 loss 1.0377  5.5 min
fold 0 ep 4/20 loss 0.9469  7.4 min
fold 0 ep 5/20 loss 0.9149  9.3 min
fold 0 ep 6/20 loss 0.8779  11.2 min
fold 0 ep 7/20 loss 0.8808  13.1 min
fold 0 ep 8/20 loss 0.8656  14.9 min
fold 0 ep 9/20 loss 0.8680  16.8 min
fold 0 ep 10/20 loss 0.8535  18.7 min
fold 0 ep 11/20 loss 0.8416  20.6 min
fold 0 ep 12/20 loss 0.8298  22.5 min
fold 0 ep 13/20 loss 0.8235  24.4 min
fold 0 ep 14/20 loss 0.8217  26.2 min
fold 0 ep 15/20 loss 0.8207  28.1 min
fold 0 ep 16/20 loss 0.8077  30.0 min
fold 0 ep 17/20 loss 0.8148  31.9 min
fold 0 ep 18/20 loss 0.8114  33.8 min
fold 0 ep 19/20 loss 0.8069  35.7 min
fold 0 ep 20/20 loss 0.8137  37.5 min
fold 0: acc 0.9903  macroF1 0.9901
fold 1 ep 1/20 loss 3.5621  1.9 min
fold 1 ep 2/20 loss 1.2016  3.8 min
fold 1 ep 3/20 loss 1.0144  5.7 min
fold 1 ep 4/20 loss 0.9533  7.5 min
fold 1 ep 5/20 loss 0.9110  9.4 min
fold 1 ep 6/20 loss 0.8878  11.3 min
fo